# Imports

In [4]:
import numpy as np
import pandas as pd

from statistics import median

import load_data
import cutpoint_analysis

import os
import pickle
import datetime
import time

import random

from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, average_precision_score, precision_recall_curve, RocCurveDisplay, accuracy_score

import mlflow
from mlflow.models import infer_signature

import great_tables as gt
from great_tables import style, loc
from great_tables import exibble

# Load data

In [6]:
imp_v1_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_normalized_includes_bSCr.csv')
imp_v2_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv')

imp_v1_df['aki_72hrs_any'] = [int(np.round(aki)) for aki in imp_v1_df['aki_72hrs_any']]
imp_v2_df['aki_72hrs_any'] = [int(np.round(aki)) for aki in imp_v2_df['aki_72hrs_any']]

# Load feature sets

In [8]:
final_feature_set_dict = dict()

for filename in os.listdir('trimmed_feature_sets'):
    if '.pickle' in filename:
        key = filename.replace('.pickle', '')
        with open('trimmed_feature_sets/' + filename, 'rb') as infile:
            final_feature_set_dict[key] = pickle.load(infile)

In [9]:
print('\n'.join(final_feature_set_dict))

auprc_ll_10_iter_logreg_only
auprc_ll_3_iter_logreg_only
auprc_med_3_iter
auprc_med_3_iter_logreg_only
auroc_ll_10_iter_logreg_only
auroc_ll_3_iter
auroc_ll_3_iter_logreg_only
auroc_med_10_iter_logreg_only
auroc_med_3_iter
auroc_med_3_iter_logreg_only


## Load model performance results as dictionaries

In [12]:
with open('model_performance_results/auprc_ll_10_iter_logreg_only_Logistic Regression_baseline_intersection.pickle', 'rb') as infile:
    test_results = pickle.load(infile)

In [13]:
test_results.keys()

dict_keys(['accuracy_cp50', 'tpr_cp50', 'tnr_cp50', 'precision_cp50', 'accuracy_cp90', 'tpr_cp90', 'tnr_cp90', 'precision_cp90', 'auroc', 'auprc'])

In [14]:
test_results

{'accuracy_cp50': 0.5174153254864281,
 'tpr_cp50': 0.8877005347593583,
 'tnr_cp50': 0.5089077282221404,
 'precision_cp50': 0.03987509007926976,
 'accuracy_cp90': 0.9006725918808551,
 'tpr_cp90': 0.5187165775401069,
 'tnr_cp90': 0.9094483351763116,
 'precision_cp90': 0.11630695443645084,
 'auroc': 0.827092503053562,
 'auprc': 0.1481110592632492}

# Define functions

## Get model performance dict and metadata from filename

In [17]:
def get_performance_info_from_filename(filename, results_dir_path='model_performance_results/', dataset='VPS/PN'):
    results_dict = dict()
    results_dict['feature_set_combo'] = 'prior' if 'prior' in filename else ('baseline_union' if 'union' in filename else 'baseline_intersection')
    results_dict['model_type'] = filename.split('_'+results_dict['feature_set_combo'])[0].split('_')[-1]
    results_dict['trim_iter_count'] = 10 if '10_iter' in filename else 3
    results_dict['logreg_only'] = 1 if 'logreg_only' in filename else 0
    results_dict['fs_metric'] = filename.split('_')[0].replace('au', '')
    results_dict['imputation'] = filename.split('_')[1]
    results_dict['dataset'] = dataset

    if '/' not in filename:
        filename = results_dir_path + filename
    with open(filename, 'rb') as infile:
        loaded_dict = pickle.load(infile)

    for key in loaded_dict.keys():
        results_dict[key.replace('precision', 'ppv')] = loaded_dict[key]

    return results_dict

## Get feature info from results dict

In [19]:


def get_features_from_results_dict(results_dict, feature_set_list_dict):
    fs_list_key = \
        'au' + results_dict['fs_metric'] + '_' + \
        results_dict['imputation'] + '_' + \
        str(results_dict['trim_iter_count']) + '_iter' + \
        ('_logreg_only' if results_dict['logreg_only']==1 else '')

    fs_list = feature_set_list_dict[fs_list_key]

    if 'union' in results_dict['feature_set_combo']:
        feature_set = []
        for fs in fs_list:
            for feature in fs:
                if feature not in feature_set:
                    feature_set.append(feature)
    else:
        feature_set = [feature for feature in fs_list[0] if all([feature in fs for fs in fs_list])]

    if 'baseline' in results_dict['feature_set_combo']:
        feature_set.append('bSCr')
    else:
        feature_set.append('prior_SCr')

    return feature_set

    # if 'baseline' in results_dict['feature_set_combo'] and 'bscr_prior' in feature_set:
    #     feature_set.remove('bscr_prior')
    # elif 'prior' in results_dict['feature_set_combo'] and 'baseline_bscr'

In [20]:
get_performance_info_from_filename('auprc_ll_10_iter_logreg_only_Logistic Regression_baseline_intersection.pickle')

{'feature_set_combo': 'baseline_intersection',
 'model_type': 'Logistic Regression',
 'trim_iter_count': 10,
 'logreg_only': 1,
 'fs_metric': 'prc',
 'imputation': 'll',
 'dataset': 'VPS/PN',
 'accuracy_cp50': 0.5174153254864281,
 'tpr_cp50': 0.8877005347593583,
 'tnr_cp50': 0.5089077282221404,
 'ppv_cp50': 0.03987509007926976,
 'accuracy_cp90': 0.9006725918808551,
 'tpr_cp90': 0.5187165775401069,
 'tnr_cp90': 0.9094483351763116,
 'ppv_cp90': 0.11630695443645084,
 'auroc': 0.827092503053562,
 'auprc': 0.1481110592632492}

In [21]:
results_dict_list = []

for f in os.listdir('model_performance_results'):
    if '.pickle' in f:
        # print(f)
        temp_results = get_performance_info_from_filename(f)
        temp_results['feature_set'] = get_features_from_results_dict(temp_results, final_feature_set_dict)
        temp_results['feature_count'] = len(temp_results['feature_set'])
        results_dict_list.append(temp_results)

results_df = pd.DataFrame(results_dict_list)

In [22]:
# results_df['bscr_feature_count'] = [len([x for x in fs if 'bSCr' in x]) for fs in results_df['feature_set']]

# results_df['bscr_feature_count'].value_counts()

In [23]:
results_df.sort_values('auprc', ascending=False).head(15)

,feature_set_combo,model_type,trim_iter_count,logreg_only,fs_metric,imputation,dataset,accuracy_cp50,tpr_cp50,tnr_cp50,ppv_cp50,accuracy_cp90,tpr_cp90,tnr_cp90,ppv_cp90,auroc,auprc,feature_set,feature_count
99,baseline_intersection,Gradient Boosting Classifier,10,1,roc,ll,VPS/PN,0.518136,0.909091,0.509153,0.040816,0.897550,0.449198,0.907851,0.100719,0.822919,0.166517,"[cal_median, fio2_min, Blood Urea Nitrogen_mea...",13
42,baseline_intersection,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
44,prior,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
20,prior,SVC poly,10,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
19,baseline_union,SVC poly,10,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
43,baseline_union,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
18,baseline_intersection,SVC poly,10,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
134,prior,Gradient Boosting Classifier,3,1,roc,ll,VPS/PN,0.516454,0.871658,0.508293,0.039136,0.899592,0.491979,0.908957,0.110444,0.811037,0.157553,"[na_max, Blood Urea Nitrogen_mean, inr_min, cr...",11
133,baseline_union,Gradient Boosting Classifier,3,1,roc,ll,VPS/PN,0.516454,0.871658,0.508293,0.039136,0.899592,0.491979,0.908957,0.110444,0.811037,0.157553,"[na_max, mag_min, Blood Urea Nitrogen_mean, in...",45
123,baseline_intersection,Gradient Boosting Classifier,3,0,roc,ll,VPS/PN,0.518857,0.893048,0.510259,0.040212,0.899111,0.486631,0.908588,0.108982,0.813095,0.156688,"[temp_max, na_median, hr_max, sbp_max, fio2_me...",72


In [24]:
results_df.sort_values('ppv_cp90', ascending=False).head(20)

,feature_set_combo,model_type,trim_iter_count,logreg_only,fs_metric,imputation,dataset,accuracy_cp50,tpr_cp50,tnr_cp50,ppv_cp50,accuracy_cp90,tpr_cp90,tnr_cp90,ppv_cp90,auroc,auprc,feature_set,feature_count
121,baseline_union,DecisionTree,3,0,roc,ll,VPS/PN,0.959044,0.149733,0.977639,0.133333,0.959044,0.149733,0.977639,0.133333,0.563695,0.039157,"[temp_max, na_median, hr_max, sbp_max, fio2_me...",84
122,prior,DecisionTree,3,0,roc,ll,VPS/PN,0.959044,0.149733,0.977639,0.133333,0.959044,0.149733,0.977639,0.133333,0.563695,0.039157,"[temp_max, na_median, hr_max, sbp_max, fio2_me...",72
96,baseline_intersection,DecisionTree,10,1,roc,ll,VPS/PN,0.961446,0.122995,0.980710,0.127778,0.961446,0.122995,0.980710,0.127778,0.551890,0.035862,"[cal_median, fio2_min, Blood Urea Nitrogen_mea...",13
98,prior,DecisionTree,10,1,roc,ll,VPS/PN,0.958083,0.144385,0.976778,0.125000,0.958083,0.144385,0.976778,0.125000,0.560582,0.037265,"[cal_median, fio2_min, Blood Urea Nitrogen_mea...",13
97,baseline_union,DecisionTree,10,1,roc,ll,VPS/PN,0.958083,0.144385,0.976778,0.125000,0.958083,0.144385,0.976778,0.125000,0.560582,0.037265,"[cl_median, Total Bilirubin_mean, cal_median, ...",84
19,baseline_union,SVC poly,10,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
18,baseline_intersection,SVC poly,10,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
44,prior,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
43,baseline_union,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84
42,baseline_intersection,SVC poly,3,1,prc,ll,VPS/PN,0.518376,0.903743,0.509522,0.040615,0.901633,0.540107,0.909940,0.121103,0.831257,0.164181,"[fio2_mean, inr_max, temp_median, cl_max, ast_...",84


In [25]:
results_df['feature_set_combo'].unique()

array(['baseline_intersection', 'baseline_union', 'prior'], dtype=object)

In [26]:
results_df.to_csv('model_performance_results.csv')

# Save feature sets as one-hot encoded dataframes

In [28]:
intermediate_feature_set_dict = dict()

for filename in os.listdir('trimmed_feature_sets'):
    if '.pickle' in filename:
        key = filename.replace('.pickle', '')
        with open('trimmed_feature_sets/' + filename, 'rb') as infile:
            intermediate_feature_set_dict[key] = pickle.load(infile)

In [29]:
print('\n'.join(intermediate_feature_set_dict.keys()))

auprc_ll_10_iter_logreg_only
auprc_ll_3_iter_logreg_only
auprc_med_3_iter
auprc_med_3_iter_logreg_only
auroc_ll_10_iter_logreg_only
auroc_ll_3_iter
auroc_ll_3_iter_logreg_only
auroc_med_10_iter_logreg_only
auroc_med_3_iter
auroc_med_3_iter_logreg_only


In [30]:
def get_feature_set_union_and_intersect(feature_set_list, verbosity=0, return_dict=False):
    union_of_sets = []

    for i, fs in enumerate(feature_set_list):
        if verbosity > 0:
            print('Set %d has length %d' % (i+1, len(fs)))
        for feature in fs:
            if feature not in union_of_sets:
                union_of_sets.append(feature)

    intersection_of_sets = [
        feature for feature in feature_set_list[0] if (
            all([feature in feature_set_list[i] for i in range(len(feature_set_list))])
        )
    ]

    if verbosity > 0:
        print('Intersection set has length %d' % len(intersection_of_sets))
        print('Union set has length %d' % len(union_of_sets))
    
    if return_dict:
        feature_set_dict = {
            'union': union_of_sets,
            'intersection': intersection_of_sets
        }

        return feature_set_dict
    
    else:
        return union_of_sets, intersection_of_sets

In [31]:
def get_final_fs_key_from_intermediate(intermediate_key):
    imp_method = 'll' if 'll' in intermediate_key else 'med'
    metric = 'prc' if 'auprc' in intermediate_key else 'roc'
    iter_count_str = '10_iter' if '10_iter' in intermediate_key else '3_iter'
    logreg_only_str = '_logreg_only' if 'logreg_only' in intermediate_key else ''
    final_key = imp_method + '_' + metric + '_' + iter_count_str + logreg_only_str
    return final_key

In [32]:
final_feature_set_dict = dict()

for key in intermediate_feature_set_dict.keys():
    final_key = get_final_fs_key_from_intermediate(key)
    union, intersect = get_feature_set_union_and_intersect(intermediate_feature_set_dict[key])
    final_feature_set_dict[final_key + '_union'] = union
    final_feature_set_dict[final_key + '_intersect'] = intersect

In [34]:
def get_model_metadata_from_key(key):
    metadata_dict = dict()
    
    metadata_dict['imputation'] = key.split('_')[0]
    metadata_dict['fs_metric'] = key.split('_')[1]
    
    metadata_dict['logreg_only'] = 1 if 'logreg_only' in key else 0
    metadata_dict['trimming_iter_count'] = 10 if '10_iter' in key else 3
    
    if 'intersect' in key:
        metadata_dict['fs_combination'] = 'intersect'
    else:
        metadata_dict['fs_combination'] = 'union'
    
    return metadata_dict

In [35]:
def get_1hc_feature_set_dict(feature_set, metadata_dict, all_possible_features):
    fs_1h_dict = metadata_dict.copy()
    fs_1h_dict['feature_count'] = len(feature_set)
    
    for feature in all_possible_features:
        if feature in feature_set:
            fs_1h_dict[feature] = 1
        else:
            fs_1h_dict[feature] = 0
            
    return fs_1h_dict

In [36]:
non_feature_cols = [
    'pn.site',
    'vps.site',
    'person_id',
    'Medical.LOS',
    'Physical.LOS',
    'hashid',
    'Female',
    'aki_12hrs',
    'aki_12hrs_any',
    'aki_72hrs',
    'aki_72hrs_any',
    'CaseIndex',
    'adt_datetime',
    'medical_dc_dt',
    'Outcome',
    'baseline_used'
]

all_possible_features = [feature for feature in imp_v1_df.columns if feature not in non_feature_cols]

In [37]:
fs_1hc_dict_list = []

for key in final_feature_set_dict.keys():
    temp_fs = final_feature_set_dict[key]
    temp_metadata_dict = get_model_metadata_from_key(key)
    temp_1hc_dict = get_1hc_feature_set_dict(temp_fs, temp_metadata_dict, all_possible_features)
    fs_1hc_dict_list.append(temp_1hc_dict)
    
fs_1hc_df = pd.DataFrame(fs_1hc_dict_list)

In [38]:
fs_1hc_df

,imputation,fs_metric,logreg_only,trimming_iter_count,fs_combination,feature_count,age,bSCr_prior,baseline_bSCr,anc_min,...,temp_mean,fio2_mean,is_immunocompromised,nephrotoxic_med_type_count,Male,nephrotoxic_med_type_count_eq_0,nephrotoxic_med_type_count_eq_1,nephrotoxic_med_type_count_eq_2,nephrotoxic_med_type_count_gt_eq_3,nephrotoxic_med_type_count_gt_0
0,ll,prc,1,10,union,83,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
1,ll,prc,1,10,intersect,83,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
2,ll,prc,1,3,union,83,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
3,ll,prc,1,3,intersect,83,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
4,med,prc,0,3,union,84,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
5,med,prc,0,3,intersect,82,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
6,med,prc,1,3,union,58,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
7,med,prc,1,3,intersect,21,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
8,ll,roc,1,10,union,83,0,0,0,1,...,0,1,0,1,0,0,0,0,0,1
9,ll,roc,1,10,intersect,12,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [39]:
fs_1hc_df.to_csv('one_hot_enc_vpspn_feature_sets.csv', index=False)

# Assess best model performance on validation and holdout sets

In [41]:
print('\n'.join(final_feature_set_dict.keys()))

ll_prc_10_iter_logreg_only_union
ll_prc_10_iter_logreg_only_intersect
ll_prc_3_iter_logreg_only_union
ll_prc_3_iter_logreg_only_intersect
med_prc_3_iter_union
med_prc_3_iter_intersect
med_prc_3_iter_logreg_only_union
med_prc_3_iter_logreg_only_intersect
ll_roc_10_iter_logreg_only_union
ll_roc_10_iter_logreg_only_intersect
ll_roc_3_iter_union
ll_roc_3_iter_intersect
ll_roc_3_iter_logreg_only_union
ll_roc_3_iter_logreg_only_intersect
med_roc_10_iter_logreg_only_union
med_roc_10_iter_logreg_only_intersect
med_roc_3_iter_union
med_roc_3_iter_intersect
med_roc_3_iter_logreg_only_union
med_roc_3_iter_logreg_only_intersect


In [42]:
final_feature_set_dict['ll_prc_10_iter_logreg_only_union']

['fio2_mean',
 'inr_max',
 'temp_median',
 'cl_max',
 'ast_median',
 'wbc_max',
 'be_min',
 'lact_median',
 'pH_min',
 'ast_min',
 'alkphos_mean',
 'gluc_min',
 'prot_min',
 'ibili_max',
 'crp_mean',
 'cl_median',
 'sbp_max',
 'cal_mean',
 'dbp_mean',
 'ast_max',
 'bicarb_min',
 'fio2_min',
 'po2_max',
 'mag_mean',
 'pco2_min',
 'spo2_median',
 'mag_max',
 'cr_max',
 'bicarb_max',
 'segs_min',
 'temp_max',
 'phos_max',
 'nephrotoxic_med_type_count_gt_0',
 'ical_max',
 'anc_min',
 'alb_max',
 'Platelet Count_min',
 'hct_max',
 'ibili_median',
 'rr_max',
 'sbp_min',
 'inr_min',
 'phos_median',
 'anc_max',
 'cbili_max',
 'Total Bilirubin_max',
 'fibr_max',
 'mag_min',
 'Blood Urea Nitrogen_mean',
 'na_median',
 'mbp_min',
 'be_max',
 'gluc_max',
 'cr_mean',
 'lact_max',
 'cal_min',
 'rr_min',
 'Total Bilirubin_min',
 'pco2_max',
 'nephrotoxic_med_type_count',
 'hr_max',
 'pt_mean',
 'hgb_mean',
 'prot_max',
 'hr_mean',
 'spo2_mean',
 'cl_min',
 'bicarb_median',
 'k_min',
 'wbc_min',
 'k_m

In [43]:
def get_datetime_obj_from_str(dt_str):
    year = int(dt_str.split('-')[0])
    month = int(dt_str.split('-')[1])
    day = int(dt_str.split('-')[2].split(' ')[0])
    try:
        hour = int(dt_str.split(' ')[1].split(':')[0])
        minute = int(dt_str.split(' ')[1].split(':')[1])
        second = int(dt_str.split(':')[-1])
    except:
        hour = 0
        minute = 0
        second = 0

    dt_obj = datetime.datetime(year, month, day, hour, minute, second)

    return dt_obj

In [44]:
df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/cohort_covariates_final.csv')

df['nephrotoxic_med_type_count_gt_0'] = [1 if x > 0 else 0 for x in df['nephrotoxic_med_type_count']]

df = df[df['aki_12hrs'].fillna(0) == 0]

df['aki_72hrs_any'] = (df['aki_72hrs'] > 0).astype(int)

df['adt_datetime_obj'] = [get_datetime_obj_from_str(dt_str) for dt_str in df['adt_datetime']]

cutoff_datetime = datetime.datetime(2016, 6, 1, 0, 0, 0)

test_df = df[df['adt_datetime_obj'] >= cutoff_datetime].copy()
analysis_df = df[df['adt_datetime_obj'] < cutoff_datetime].copy()

In [45]:
train_df, val_df = train_test_split(analysis_df, 
                                    test_size=0.2,
                                    stratify=analysis_df['aki_72hrs_any'],
                                    random_state=343)

In [46]:
print('\n'.join([col for col in train_df.columns if 'neph' in col]))

nephrotoxic_med_type_count
nephrotoxic_med_type_count_gt_0


## Select relevant features only

In [48]:
train_df = train_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union'] + ['aki_72hrs_any']]
val_df = val_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union'] + ['aki_72hrs_any']]
test_df = test_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union'] + ['aki_72hrs_any']]

## Impute with median

In [50]:
for feature in train_df.columns:
    if feature != 'aki_72hrs_any':
        median_val = median(train_df[feature].dropna())

        train_df[feature] = train_df[feature].fillna(median_val)
        val_df[feature] = val_df[feature].fillna(median_val)
        test_df[feature] = test_df[feature].fillna(median_val)

## Normalize

In [52]:
for feature in train_df.columns:
    if feature != 'aki_72hrs_any':
        min_val = train_df[feature].min()
        max_val = train_df[feature].max()

        train_df[feature] = [(x - min_val) / (max_val - min_val) for x in train_df[feature]]
        val_df[feature] = [(x - min_val) / (max_val - min_val) for x in val_df[feature]]
        test_df[feature] = [(x - min_val) / (max_val - min_val) for x in test_df[feature]]

## Train model

In [54]:
X_train = train_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union']].to_numpy()
y_train = train_df['aki_72hrs_any'].to_numpy()

X_test = test_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union']].to_numpy()
y_test = test_df['aki_72hrs_any'].to_numpy()

X_val = val_df[final_feature_set_dict['ll_prc_10_iter_logreg_only_union']].to_numpy()
y_val = val_df['aki_72hrs_any'].to_numpy()

In [55]:
model = LogisticRegression(class_weight='balanced', max_iter=10000, random_state=343)

In [56]:
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=10000, random_state=343)

In [57]:
y_score_test = model.predict_proba(X_test)
y_score_val = model.predict_proba(X_val)

In [58]:
val_scores_df = pd.DataFrame({
    'aki_72hrs_any': y_val,
    'risk_score': y_score_val[:,1]
})

val_scores_df.to_csv('vps_pn_model_val_scores_and_actual.csv', index=False)

test_scores_df = pd.DataFrame({
    'aki_72hrs_any': y_test,
    'risk_score': y_score_test[:,1]
})

test_scores_df.to_csv('vps_pn_model_test_scores_and_actual.csv', index=False)

In [59]:
def get_performance_results(y_test, y_score):
    print('N = %d' % len(y_test))
    print('%% with AKI: %.1f\n' % (100 * y_test.sum() / len(y_test)))
    
    roc_auc = roc_auc_score(y_test, y_score[:,1])
    print('AUROC = %.2f' % roc_auc)
    pr_auc = average_precision_score(y_test, y_score[:,1])
    print('AUPRC = %.2f\n' % pr_auc)

    cp50_dict = cutpoint_analysis.get_results_at_cutpoint(y_test, y_score[:,1], cutpoint=0.5)
    print('Predicted positive at cutpoint 50: %d (%.1f%%)' % (cp50_dict['y_pred_positive_count'], 100 * cp50_dict['y_pred_positive_count'] / len(y_test)))
    print('Sensitivity at cutpoint 50: %.2f' % cp50_dict['metrics']['tpr'])
    print('Specificity at cutpoint 50: %.2f' % cp50_dict['metrics']['tnr'])
    print('PPV at cutpoint 50: %.2f\n' % cp50_dict['metrics']['precision'])

    cp90_dict = cutpoint_analysis.get_results_at_cutpoint(y_test, y_score[:,1], cutpoint=0.9)
    print('Predicted positive at cutpoint 90: %d (%.1f%%)' % (cp90_dict['y_pred_positive_count'], 100 * cp90_dict['y_pred_positive_count'] / len(y_test)))
    print('Sensitivity at cutpoint 90: %.2f' % cp90_dict['metrics']['tpr'])
    print('Specificity at cutpoint 90: %.2f' % cp90_dict['metrics']['tnr'])
    print('PPV at cutpoint 90: %.2f' % cp90_dict['metrics']['precision'])

In [60]:
cp50_dict = cutpoint_analysis.get_results_at_cutpoint(y_train, model.predict_proba(X_train)[:,1], cutpoint=0.5)
cp90_dict = cutpoint_analysis.get_results_at_cutpoint(y_train, model.predict_proba(X_train)[:,1], cutpoint=0.9)

cutpoint50 = cp50_dict['pred_threshold']
cutpoint90 = cp90_dict['pred_threshold']

cutpoint_dict = {
    50: cutpoint50,
    90: cutpoint90
}

cutpoint_dict

{50: 0.3406982421875, 90: 0.631103515625}

In [61]:
with open('vps_pn_cutpoints_from_train_set.pickle', 'wb') as outfile:
    pickle.dump(cutpoint_dict, outfile)

# Get performance results with cutpoints from train set

In [ ]:
val_scores_df['risk_score'] = [model.predict_proba(x.reshape(1, -1))[:,1] for x in train_cohort_ll[modeling_feature_set].to_numpy()]

In [113]:
# val_scores_df['pred_cp50'] = 

val_scores_df['pred_cp50'] = [score > cutpoint50 for score in val_scores_df['risk_score']]
val_scores_df['pred_cp90'] = [score > cutpoint90 for score in val_scores_df['risk_score']]

test_scores_df['pred_cp50'] = [score > cutpoint50 for score in test_scores_df['risk_score']]
test_scores_df['pred_cp90'] = [score > cutpoint90 for score in test_scores_df['risk_score']]

In [111]:
def get_pred_type_counts(y_actual, y_pred):
    tp = ((y_actual==1) & (y_pred==1)).sum()
    fp = ((y_actual==0) & (y_pred==1)).sum()
    tn = ((y_actual==0) & (y_pred==0)).sum()
    fn = ((y_actual==1) & (y_pred==0)).sum()
    return tp, fp, tn, fn

In [115]:
print(
    '[VAL - CUTPOINT 50]\nTP = %d\nFP = %d\nTN = %d\nFN = %d' %  \
        get_pred_type_counts(val_scores_df['aki_72hrs_any'].to_numpy(), val_scores_df['pred_cp50'].to_numpy())
)

[VAL - CUTPOINT 50]
TP = 162
FP = 3973
TN = 4166
FN = 25


In [117]:
print(
    '[VAL - CUTPOINT 90]\nTP = %d\nFP = %d\nTN = %d\nFN = %d' %  \
        get_pred_type_counts(val_scores_df['aki_72hrs_any'].to_numpy(), val_scores_df['pred_cp90'].to_numpy())
)

[VAL - CUTPOINT 90]
TP = 89
FP = 765
TN = 7374
FN = 98


In [119]:
print(
    '[TEST - CUTPOINT 50]\nTP = %d\nFP = %d\nTN = %d\nFN = %d' %  \
        get_pred_type_counts(test_scores_df['aki_72hrs_any'].to_numpy(), test_scores_df['pred_cp50'].to_numpy())
)

[TEST - CUTPOINT 50]
TP = 253
FP = 6671
TN = 7290
FN = 50


In [121]:
print(
    '[TEST - CUTPOINT 90]\nTP = %d\nFP = %d\nTN = %d\nFN = %d' %  \
        get_pred_type_counts(test_scores_df['aki_72hrs_any'].to_numpy(), test_scores_df['pred_cp90'].to_numpy())
)

[TEST - CUTPOINT 90]
TP = 141
FP = 1277
TN = 12684
FN = 162


In [77]:
get_performance_results(y_val, y_score_val)

N = 8326
% with AKI: 2.2

AUROC = 0.80
AUPRC = 0.13

Predicted positive at cutpoint 50: 4163 (50.0%)
Sensitivity at cutpoint 50: 0.87
Specificity at cutpoint 50: 0.51
PPV at cutpoint 50: 0.04

Predicted positive at cutpoint 90: 833 (10.0%)
Sensitivity at cutpoint 90: 0.47
Specificity at cutpoint 90: 0.91
PPV at cutpoint 90: 0.10


In [78]:
get_performance_results(y_test, y_score_test)

N = 14264
% with AKI: 2.1

AUROC = 0.78
AUPRC = 0.11

Predicted positive at cutpoint 50: 7130 (50.0%)
Sensitivity at cutpoint 50: 0.84
Specificity at cutpoint 50: 0.51
PPV at cutpoint 50: 0.04

Predicted positive at cutpoint 90: 1427 (10.0%)
Sensitivity at cutpoint 90: 0.47
Specificity at cutpoint 90: 0.91
PPV at cutpoint 90: 0.10


## Plot ROC and PRC

In [85]:
def get_roc_and_prc_df(y_test, y_score):

    fpr, tpr, thresholds = roc_curve(y_test, y_score[:,1])
    
    roc_df = pd.DataFrame(
        {
            'model': ['vps_pn_test' for _ in range(len(fpr))],
            'fpr': fpr,
            'tpr': tpr
        }
    )

    fpr, tpr, thresholds = precision_recall_curve(y_test, y_score[:,1])
    
    prc_df = pd.DataFrame(
        {
            'model': ['vps_pn_test' for _ in range(len(fpr))],
            'fpr': fpr,
            'tpr': tpr
        }
    )

    return roc_df, prc_df

In [86]:
roc_test_df, prc_test_df = get_roc_and_prc_df(y_test, y_score_test)

In [88]:
roc_test_df.to_csv('best_model_test_roc_df.csv', index=False)
prc_test_df.to_csv('best_model_test_prc_df.csv', index=False)

In [89]:
roc_val_df, prc_val_df = get_roc_and_prc_df(y_val, y_score_val)

In [90]:
roc_val_df.to_csv('best_model_val_roc_df.csv', index=False)
prc_val_df.to_csv('best_model_val_prc_df.csv', index=False)